In [34]:
%load_ext autoreload
%autoreload 2

import sys, os
sys.path.append(os.path.abspath(".."))   # so Python finds the 'src' folder

import pandas as pd
from src.model import Problem, State, Stop
from src import informed

The autoreload extension is already loaded. To reload it, use:
  %reload_ext autoreload


# 4. Informed Search: Greedy and A*

**Goal:** build the route of each vehicle with informed search and compare Greedy best-first with A* (two heuristics), counting expanded nodes.

## Problem formulation

Every item follows the flow `producer -> warehouse -> customer`, so the trip of one vehicle has four phases:

`depot -> producers -> warehouse (unload + load) -> customers -> depot`

| Element | Definition |
|---|---|
| **State** | `(current location, producers still to visit, warehouse done?, customers still to visit)` |
| **Actions** | Go to one location allowed in the current phase: a pending producer (pickup), the warehouse (unload + load), a pending customer (deliver), or back to the depot |
| **Step cost** | Distance in km between the two locations, from `prob.distance` |
| **Goal** | All producers and customers visited, warehouse done, vehicle back at the depot |
| **Initial state** | Vehicle at the depot with everything pending |

The items are first split between the two vehicles by capacity (Vehicle 1: 150 kg, Vehicle 2: 200 kg), then each vehicle is solved with its own search.

## Algorithms

* **Greedy best-first:** expands the node with the lowest `h(n)`. Fast, but not optimal.
* **A\*:** expands the node with the lowest `f(n) = g(n) + h(n)`. Optimal if `h` is admissible.

## Heuristics

* **h1:** distance from the current location to the nearest location that still has to be visited.
* **h2:** sum, over every location still to be visited, of the cheapest edge that enters it.

Both never overestimate the remaining cost (every remaining location must be entered at least once through an edge at least as long as its cheapest incoming edge), so they are admissible and A* returns the optimal route for the given item split.

### Load the problem and split the items between vehicles

In [35]:
prob = Problem.from_csv("../data_tiny")

assignment = informed.split_items(prob)
for vid, ids in assignment.items():
    kg = sum(prob.req_map[i].weight_kg for i in ids)
    print(f"Vehicle {vid}: {len(ids)} items, {kg:.1f} kg")

Vehicle 1: 24 items, 148.7 kg
Vehicle 2: 16 items, 121.3 kg


### Compare Greedy and A* (h1, h2)

Iteration 1-2 setting: flow and capacity only (no time windows yet).

In [36]:
df, states = informed.run_all(prob, assignment=assignment)
df

,algorithm,heuristic,cost_km,expanded_nodes,generated_nodes,seconds,status,feasible
0,greedy,h1,464.80,121,254,0.0014,solved,True
1,greedy,h2,470.46,20,42,0.0001,solved,True
2,astar,h1,464.80,125,251,0.0009,solved,True
3,astar,h2,464.80,106,212,0.0008,solved,True


### Best route found (A* with h1)

In [37]:
best = states[("astar", "h1")]
for vid, route in best.routes.items():
    print(f"\nVehicle {vid}")
    for s in route:
        print(f"  {s.action:8s} at location {s.location_id:>3}  items={list(s.items)}")

print("\nTotal cost (km):", round(prob.calculate_cost(best), 2))
print("Feasible (flow + capacity):", prob.is_feasible(best, check_capacity=True))


Vehicle 1
  pickup   at location   1  items=[2, 5, 10, 13, 19, 20, 21, 25]
  pickup   at location   2  items=[1, 7, 12, 15, 16, 18]
  pickup   at location   3  items=[3, 4, 6, 8, 9, 11, 14, 17, 22, 24]
  unload   at location   4  items=[1, 2, 3, 4, 5, 6, 7, 8, 9, 10, 11, 12, 13, 14, 15, 16, 17, 18, 19, 20, 21, 22, 24, 25]
  load     at location   4  items=[1, 2, 3, 4, 5, 6, 7, 8, 9, 10, 11, 12, 13, 14, 15, 16, 17, 18, 19, 20, 21, 22, 24, 25]
  deliver  at location   6  items=[7, 8, 9, 10, 22, 24]
  deliver  at location   8  items=[5, 6]
  deliver  at location   9  items=[18, 19, 20, 21, 25]
  deliver  at location   7  items=[1, 2, 3, 4, 11, 12, 13, 16, 17]
  deliver  at location   5  items=[14, 15]

Vehicle 2
  pickup   at location   1  items=[31, 32, 34, 37, 40]
  pickup   at location   2  items=[23, 27, 28, 30, 33, 36, 38, 39]
  pickup   at location   3  items=[26, 29, 35]
  unload   at location   4  items=[23, 26, 27, 28, 29, 30, 31, 32, 33, 34, 35, 36, 37, 38, 39, 40]
  load     a

### With time windows

Pickup windows (availability at the producer) and delivery windows (customer deadline). Arriving early means waiting, arriving late discards the branch.
`start_time` is when the vehicles leave the depot; adjust it to what the `rules` table says.

In [38]:
df_t, states_t = informed.run_all(prob, assignment=assignment, use_time=True)
df_t

,algorithm,heuristic,cost_km,expanded_nodes,generated_nodes,seconds,status,feasible
0,greedy,h1,NaN,21,21,0.0002,no_solution,False
1,greedy,h2,NaN,21,21,0.0002,no_solution,False
2,astar,h1,NaN,21,21,0.0002,no_solution,False
3,astar,h2,NaN,21,21,0.0002,no_solution,False


## Conclusions

*Fill in after running the notebook:*

* Which algorithm expanded fewer nodes?
* Did A* return a cheaper route than Greedy? By how much?
* Was h2 better informed than h1 (fewer expanded nodes)?
* What happened with the time windows (still solved? higher cost? no solution)?

In [39]:
depot = prob.depot_loc
prods = sorted({r.pickup_loc for r in prob.requests})
custs = sorted({r.delivery_loc for r in prob.requests})
has = lambda a, b: a == b or (a, b) in prob.distance

print("depot:", depot, "| producers:", prods, "| customers:", custs)
print("distance entries:", len(prob.distance))
print("depot->producer ok:   ", [has(depot, p) for p in prods])
print("producer->depot ok:   ", [has(p, depot) for p in prods])
print("depot->customer ok:   ", [has(depot, c) for c in custs])
print("customer->depot ok:   ", [has(c, depot) for c in custs])
print("producer->producer missing:", sum(not has(a, b) for a in prods for b in prods))
print("customer->customer missing:", sum(not has(a, b) for a in custs for b in custs))
print("sample keys:", list(prob.distance.items())[:3])

state, res = informed.solve(prob, "astar", "h1", assignment=assignment)
for r in res:
    print("vehicle", r.vehicle_id, r.status, "expanded", r.expanded, "generated", r.generated)

depot: 4 | producers: [1, 2, 3] | customers: [5, 6, 7, 8, 9]
distance entries: 72
depot->producer ok:    [True, True, True]
producer->depot ok:    [True, True, True]
depot->customer ok:    [True, True, True, True, True]
customer->depot ok:    [True, True, True, True, True]
producer->producer missing: 0
customer->customer missing: 0
sample keys: [((1, 2), 23.59), ((1, 3), 39.3), ((1, 4), 30.6)]
vehicle 1 solved expanded 83 generated 178
vehicle 2 solved expanded 42 generated 73


In [40]:
g = df[(df.algorithm == "greedy") & (df.heuristic == "h1")].cost_km.iloc[0]
a = df[(df.algorithm == "astar") & (df.heuristic == "h1")].cost_km.iloc[0]
print(f"Greedy: {g:.2f} km | A*: {a:.2f} km | saving: {g - a:.2f} km ({(g - a) / g * 100:.1f}%)")

Greedy: 464.80 km | A*: 464.80 km | saving: 0.00 km (0.0%)


In [41]:
a = df[df.algorithm == "astar"].set_index("heuristic")

if (a.status != "solved").any():
    print("A* did not solve the problem: fix that first (see the 'status' column).")
else:
    n1, n2 = a.loc["h1", "expanded_nodes"], a.loc["h2", "expanded_nodes"]
    c1, c2 = a.loc["h1", "cost_km"], a.loc["h2", "cost_km"]
    print(f"A* with h1: {n1} nodes, {c1:.2f} km")
    print(f"A* with h2: {n2} nodes, {c2:.2f} km")

    if n2 < n1:
        print(f"-> h2 is better informed: {n1 - n2} fewer nodes ({(n1 - n2) / n1 * 100:.1f}%).")
    elif n1 < n2:
        print(f"-> h1 is better informed: {n2 - n1} fewer nodes ({(n2 - n1) / n2 * 100:.1f}%).")
    else:
        print("-> Same number of nodes: neither heuristic is better informed on this data.")

    if abs(c1 - c2) > 1e-6:
        print("WARNING: the costs differ, so one heuristic is not admissible. Tell me.")

A* with h1: 125 nodes, 464.80 km
A* with h2: 106 nodes, 464.80 km
-> h2 is better informed: 19 fewer nodes (15.2%).


In [42]:
m = df.merge(df_t, on=["algorithm", "heuristic"], suffixes=("_no_time", "_time"))
m["cost_change_km"] = (m["cost_km_time"] - m["cost_km_no_time"]).round(3)

print(m[["algorithm", "heuristic", "status_no_time", "status_time",
         "cost_km_no_time", "cost_km_time", "cost_change_km"]].to_string(index=False))
print()

for _, r in m.iterrows():
    name = f"{r.algorithm} {r.heuristic}"
    if r.status_no_time != "solved":
        print(f"{name}: no route even without time windows, so nothing to compare.")
    elif r.status_time != "solved":
        print(f"{name}: NO SOLUTION with time windows ({r.status_time}).")
    elif abs(r.cost_change_km) < 1e-6:
        print(f"{name}: still solved, same cost.")
    else:
        print(f"{name}: still solved, cost went up by {r.cost_change_km:.2f} km.")

algorithm heuristic status_no_time status_time  cost_km_no_time  cost_km_time  cost_change_km
   greedy        h1         solved no_solution           464.80           NaN             NaN
   greedy        h2         solved no_solution           470.46           NaN             NaN
    astar        h1         solved no_solution           464.80           NaN             NaN
    astar        h2         solved no_solution           464.80           NaN             NaN

greedy h1: NO SOLUTION with time windows (no_solution).
greedy h2: NO SOLUTION with time windows (no_solution).
astar h1: NO SOLUTION with time windows (no_solution).
astar h2: NO SOLUTION with time windows (no_solution).
